# m-stash Quickstart (Python)

Walks through the same flow as [quickstart.md](../../quickstart.md): sign up, publish a public
profile, create private and public stashes, read them back as the owner and as an anonymous
visitor, then run one live query over the WebSocket API.

Requirements: `pip install -r requirements.txt`, and a running m-stash instance (defaults to
`http://localhost:4000`).

In [ ]:
import os
import random
import string
import requests

API_URL = os.environ.get("API_URL", "http://localhost:4000")


def random_suffix(n=6):
    return "".join(random.choices(string.ascii_lowercase + string.digits, k=n))


def api(path, method="GET", token=None, json_body=None):
    headers = {}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    response = requests.request(method, f"{API_URL}{path}", json=json_body, headers=headers, timeout=10)
    if not response.ok:
        raise RuntimeError(f"{method} {path} -> {response.status_code}: {response.text}")
    return response.json() if response.content else {}


suffix = random_suffix()
email = f"demo-{suffix}@example.com"
password = "use-a-long-unique-password"
handle = f"demo-{suffix}"

print("API_URL:", API_URL)
print("email:", email)
print("handle:", handle)

## 1. Sign up

Creates the account and returns a session token plus the new user's id.

In [ ]:
signup = api("/v1/auth/signup", method="POST", json_body={"email": email, "password": password})
token = signup["token"]
user_id = signup["user"]["id"]
print(f"signed up as {signup['user']['email']} (id: {user_id})")

## 2. Publish a public profile

`_id` is derived from the signed-in user; clients cannot set it. `handle` must be 3-32 lowercase
letters, numbers, hyphens, or underscores.

In [ ]:
api(
    "/v1/db/profiles/insertOne",
    method="POST",
    token=token,
    json_body={
        "payload": {
            "handle": handle,
            "displayName": "Demo User",
            "bio": "Created by the Python quickstart notebook.",
            "avatarURL": "https://example.com/avatar.jpg",
            "links": [{"label": "Website", "url": "https://example.com"}],
            "isPublic": True,
        }
    },
)
print("profile published:", handle)

## 3. Read the public profile anonymously

No token required. Only the safe, allowlisted fields are returned.

In [ ]:
public_profile = api(f"/v1/public/profiles/{handle}")
public_profile["data"]

## 4. Create private and public stashes

The gateway assigns `ownerId`, `createdAt`, and `updatedAt`; clients cannot supply their own
timestamps and any `ownerId` must match the signed-in user.

In [ ]:
api(
    "/v1/db/stashes/insertOne",
    method="POST",
    token=token,
    json_body={
        "payload": {
            "title": "Private draft",
            "content": "Only the owner can read this.",
            "isPublic": False,
        }
    },
)

api(
    "/v1/db/stashes/insertOne",
    method="POST",
    token=token,
    json_body={
        "payload": {
            "title": "Hello, public web",
            "summary": "A visible post.",
            "content": "This is shared intentionally.",
            "tags": ["intro"],
            "isPublic": True,
        }
    },
)
print("created one private and one public stash")

## 5. Compare owner vs. anonymous visibility

The authenticated `find` returns both stashes; the public route returns only the published one.

In [ ]:
mine = api(
    "/v1/db/stashes/find",
    method="POST",
    token=token,
    json_body={"query": {"ownerId": user_id}},
)
print("owner sees:", [(s["title"], s.get("isPublic", False)) for s in mine["data"]])

public_stashes = api(f"/v1/public/profiles/{handle}/stashes")
print("anonymous visitor sees:", [s["title"] for s in public_stashes["data"]])

## 6. Optional: live query over WebSocket

Each WebSocket message uses the same JSON shape, security policy, and actions as the HTTP API.
Uses top-level `await`, which Jupyter supports natively in code cells.

In [ ]:
import json
import websockets

ws_url = API_URL.replace("http", "ws", 1) + "/v1/ws/stashes"

async with websockets.connect(ws_url, additional_headers={"Authorization": f"Bearer {token}"}) as ws:
    await ws.send(json.dumps({"action": "find", "query": {"isPublic": False}}))
    response = await ws.recv()

json.loads(response)